In [11]:
import pandas as pd
import re
import pandas as pd
import numpy as np

In [5]:
pd.set_option("display.max_rows", None)

In [43]:
import re
import pandas as pd


def load_weather_lines(path: str) -> list[str]:
    with open(path, "r", encoding="utf-8") as f:
        return [
            line.strip()
            for line in f
            if line.strip() and not line.strip().startswith("#")
        ]


def parse_file_timestamp(ts_str: str):
    try:
        return pd.to_datetime(ts_str, format="%Y%m%d%H%M", utc=True, errors="raise")
    except Exception:
        return pd.NaT


def ddhh_to_timestamp(ddhh: str, base_ts: pd.Timestamp):
    if pd.isna(ddhh) or pd.isna(base_ts):
        return pd.NaT

    try:
        day = int(ddhh[:2])
        hour = int(ddhh[2:])

        if hour == 24:
            ts = (
                pd.Timestamp(
                    year=base_ts.year,
                    month=base_ts.month,
                    day=day,
                    hour=0,
                    minute=0
                ).tz_localize("UTC")
                + pd.Timedelta(days=1)
            )
        else:
            ts = pd.Timestamp(
                year=base_ts.year,
                month=base_ts.month,
                day=day,
                hour=hour,
                minute=0
            ).tz_localize("UTC")

        return ts

    except Exception:
        return pd.NaT


def taf_valid_to_timestamp(ddhh: str, base_ts: pd.Timestamp, valid_from: pd.Timestamp):
    if pd.isna(ddhh) or pd.isna(base_ts) or pd.isna(valid_from):
        return pd.NaT

    try:
        day = int(ddhh[:2])
        hour = int(ddhh[2:])

        if hour == 24:
            candidate = (
                pd.Timestamp(
                    year=base_ts.year,
                    month=base_ts.month,
                    day=day,
                    hour=0,
                    minute=0
                ).tz_localize("UTC")
                + pd.Timedelta(days=1)
            )
        else:
            candidate = pd.Timestamp(
                year=base_ts.year,
                month=base_ts.month,
                day=day,
                hour=hour,
                minute=0
            ).tz_localize("UTC")

        if candidate < valid_from:
            next_month = base_ts + pd.DateOffset(months=1)

            if hour == 24:
                candidate = (
                    pd.Timestamp(
                        year=next_month.year,
                        month=next_month.month,
                        day=day,
                        hour=0,
                        minute=0
                    ).tz_localize("UTC")
                    + pd.Timedelta(days=1)
                )
            else:
                candidate = pd.Timestamp(
                    year=next_month.year,
                    month=next_month.month,
                    day=day,
                    hour=hour,
                    minute=0
                ).tz_localize("UTC")

        return candidate

    except Exception:
        return pd.NaT


def parse_weather_line(line: str):
    parts = line.split(maxsplit=2)
    if len(parts) < 3:
        return None

    ts_str, report_type, body = parts[0], parts[1], parts[2]

    if report_type not in {"METAR", "TAF"}:
        return None

    file_ts = parse_file_timestamp(ts_str)

    station_match = re.search(r"^(?:(?:AMD|COR|RTD|NIL|CNL)\s+)*([A-Z]{4})\b", body)
    station = station_match.group(1) if station_match else None

    rec = {
        "raw_line": line,
        "file_ts": file_ts,
        "report_type": report_type,
        "station": station,
        "report_text": f"{report_type} {body}",
        "issued_at": file_ts,
        "valid_from": pd.NaT,
        "valid_to": pd.NaT,
    }

    if report_type == "TAF":
        m = re.search(r"\b(\d{4})/(\d{4})\b", body)
        if m:
            from_tok = m.group(1)
            to_tok = m.group(2)

            vf = ddhh_to_timestamp(from_tok, file_ts)
            vt = taf_valid_to_timestamp(to_tok, file_ts, vf)

            rec["valid_from"] = vf
            rec["valid_to"] = vt

    return rec


def load_weather_dataframe(path: str) -> pd.DataFrame:
    lines = load_weather_lines(path)
    records = []

    for line in lines:
        rec = parse_weather_line(line)
        if rec is not None:
            records.append(rec)

    df = pd.DataFrame(records)

    for col in ["file_ts", "issued_at", "valid_from", "valid_to"]:
        df[col] = pd.to_datetime(df[col], utc=True, errors="coerce")

    df = df.sort_values(["station", "issued_at", "report_type"], kind="mergesort").reset_index(drop=True)
    return df


def ensure_utc_timestamp(t):
    t = pd.Timestamp(t)
    if t.tz is None:
        return t.tz_localize("UTC")
    return t.tz_convert("UTC")


def get_active_metar(weather_df: pd.DataFrame, t, station: str = None):
    t = ensure_utc_timestamp(t)

    q = weather_df[weather_df["report_type"] == "METAR"].copy()
    if station is not None:
        q = q[q["station"] == station]

    q = q[
        (q["issued_at"].notna()) &
        (q["issued_at"] <= t) &
        (q["issued_at"] >= t - pd.Timedelta(minutes=30))
    ]

    if q.empty:
        return None

    return q.sort_values("issued_at").iloc[-1]


def get_active_taf(weather_df: pd.DataFrame, t, station: str = None):
    t = ensure_utc_timestamp(t)

    q = weather_df[weather_df["report_type"] == "TAF"].copy()
    if station is not None:
        q = q[q["station"] == station]

    q = q[
        (q["valid_from"].notna()) &
        (q["valid_to"].notna()) &
        (q["valid_from"] <= t) &
        (q["valid_to"] >= t)
    ]

    if q.empty:
        return None

    return q.sort_values("issued_at").iloc[-1]


def build_weather_prompt(weather_df: pd.DataFrame, t, station: str = None) -> str:
    metar = get_active_metar(weather_df, t, station=station)
    taf = get_active_taf(weather_df, t, station=station)

    metar_text = metar["raw_line"] if metar is not None else "N/A"
    taf_text = taf["raw_line"] if taf is not None else "N/A"

    return f"METAR in effect: {metar_text}\n\nTAF in effect: {taf_text}"

In [ ]:
weather_path = "/Users/YGT/ist-airport-decision-support-system/data/bronze/weather/allmetartaf.txt"

In [ ]:
weather_df = load_weather_dataframe(weather_path)

report_type
METAR    16319
TAF       1422
Name: count, dtype: int64


In [47]:
def attach_weather_prompt(row, weather_df):
    station = row["station"]
    t = row["sample_time"]

    metar = get_active_metar(weather_df, t, station=station)
    taf = get_active_taf(weather_df, t, station=station)

    metar_text = metar["raw_line"] if metar is not None else "N/A"
    taf_text = taf["raw_line"] if taf is not None else "N/A"

    return pd.Series({
        "metar_in_effect": metar_text,
        "taf_in_effect": taf_text,
        "weather_prompt": f"METAR in effect: {metar_text}\n\nTAF in effect: {taf_text}"
    })

In [48]:
flight_df = pd.read_parquet("/Users/YGT/ist-airport-decision-support-system/data/gold/ist_flights_labeled_v6.parquet")

In [50]:
flight_df["actual_entry_time"] = pd.to_datetime(flight_df["actual_entry_time"], unit="ms", utc=True)

def attach_weather(row, weather_df):
    t = row["actual_entry_time"]
    station = row["dest_code_icao"]

    metar = get_active_metar(weather_df, t, station=station)
    taf = get_active_taf(weather_df, t, station=station)

    metar_text = metar["raw_line"] if metar is not None else "N/A"
    taf_text = taf["raw_line"] if taf is not None else "N/A"

    return f"METAR in effect: {metar_text}\n\nTAF in effect: {taf_text}"

flight_df["weather_prompt"] = flight_df.apply(lambda row: attach_weather(row, weather_df), axis=1)


print(f"N/A sayısı: {flight_df['weather_prompt'].str.contains('N/A').sum()}")
print(flight_df[["callsign_code_iata", "actual_entry_time", "weather_prompt"]].head())

N/A sayısı: 1
  callsign_code_iata         actual_entry_time  \
0             IA 223 2025-07-31 17:29:23+00:00   
1             SV 261 2025-07-31 17:41:16+00:00   
2             ET 722 2025-07-31 18:14:19+00:00   
3            TK 1098 2025-07-31 18:11:34+00:00   
4             A3 434 2025-07-31 18:21:48+00:00   

                                      weather_prompt  
0  METAR in effect: 202507311720 METAR LTFM 31172...  
1  METAR in effect: 202507311720 METAR LTFM 31172...  
2  METAR in effect: 202507311750 METAR LTFM 31175...  
3  METAR in effect: 202507311750 METAR LTFM 31175...  
4  METAR in effect: 202507311820 METAR LTFM 31182...  


In [51]:
flight_df.head()

,date,day_of_week,hour_ist,hex_icao,aircraft_type,aircraft_registration,airline_name_english,callsign_code_iata,callsign_code_icao,airline_iata,...,post_points,holding_detected,holding_dist_std,holding_n_reversals,entry_source,entry_segment_source,is_matched,is_plausible_label,match_quality_flag,weather_prompt
0,2025-07-31,Thursday,21,728678,Airbus A320,YI-ASX,I A W,IA 223,IAW223,IA,...,1.0,True,13.936035,2,first_point_inside_entry_band,full_lookback_window_no_gap,True,True,good_match_high_conf,METAR in effect: 202507311720 METAR LTFM 31172...
1,2025-07-31,Thursday,21,7114EF,Airbus A321,UNKNOWN,Saudi Arabian,SV 261,SVA261,SV,...,1.0,False,14.526094,0,first_point_inside_entry_band,full_lookback_window_no_gap,True,True,good_match_high_conf,METAR in effect: 202507311720 METAR LTFM 31172...
2,2025-07-31,Thursday,21,040184,Boeing 737,ET-AXG,Ethiopian,ET 722,ETH722,ET,...,1.0,True,13.835181,2,first_point_inside_entry_band,full_lookback_window_no_gap,True,True,good_match_high_conf,METAR in effect: 202507311750 METAR LTFM 31175...
3,2025-07-31,Thursday,21,4BA91A,Airbus A321,TC-JHZ,Turkish,TK 1098,THY9GS,TK,...,1.0,False,5.331014,0,first_point_inside_entry_band,full_lookback_window_no_gap,True,True,good_match_high_conf,METAR in effect: 202507311750 METAR LTFM 31175...
4,2025-07-31,Thursday,21,4691C2,Airbus A320,SX-DNB,Aegean,A3 434,AEE434,A3,...,1.0,False,5.460646,0,first_point_inside_entry_band,full_lookback_window_no_gap,True,True,good_match_high_conf,METAR in effect: 202507311820 METAR LTFM 31182...


In [53]:
flight_df = flight_df[["hex_icao",
                        "airline_name_english",
                         "callsign_code_iata",
                         "callsign_code_icao",
                         "haul",
                         "dep_code_iata",
                         "dep_code_icao",
                         "dep_name_english",
                         "dep_lat",
                         "dep_lon",
                         "dep_altitude",
                         "dest_code_iata",
                         "dest_code_icao",
                         "dest_name_english",
                         "dest_lat",
                         "dest_lon",
                         "dest_altitude",
                         "distance",
                         "actual_entry_time",
                         "arr_sched_time_utc",
                         "date",
                         "day_of_week",
                         "aircraft_type",
                         "aircraft_registration",
                         "wake_turbulence_cat",
                         "post_terminal_duration_min",
                         "weather_prompt"
                         ]].copy()

In [54]:
flight_df.columns

Index(['hex_icao', 'airline_name_english', 'callsign_code_iata',
       'callsign_code_icao', 'haul', 'dep_code_iata', 'dep_code_icao',
       'dep_name_english', 'dep_lat', 'dep_lon', 'dep_altitude',
       'dest_code_iata', 'dest_code_icao', 'dest_name_english', 'dest_lat',
       'dest_lon', 'dest_altitude', 'distance', 'actual_entry_time',
       'arr_sched_time_utc', 'date', 'day_of_week', 'aircraft_type',
       'aircraft_registration', 'wake_turbulence_cat',
       'post_terminal_duration_min', 'weather_prompt'],
      dtype='str')

In [ ]:
flight_df.to_parquet(
    "/Users/YGT/ist-airport-decision-support-system/data/train/ist_flights_weather.parquet",
    index=False
)

In [56]:
flight_df["weather_prompt"].str.len().describe()

count    30883.000000
mean       201.702004
std         43.890357
min         98.000000
25%        163.000000
50%        180.000000
75%        243.000000
max        347.000000
Name: weather_prompt, dtype: float64

In [57]:
flight_df.sample(5)[["actual_entry_time","weather_prompt"]]

,actual_entry_time,weather_prompt
11865,2025-06-28 08:05:17+00:00,METAR in effect: 202506280750 METAR LTFM 28075...
1164,2025-08-07 05:44:42+00:00,METAR in effect: 202508070520 METAR LTFM 07052...
30040,2025-09-22 00:53:23+00:00,METAR in effect: 202509220050 METAR LTFM 22005...
24785,2025-10-11 16:35:22+00:00,METAR in effect: 202510111620 METAR LTFM 11162...
29845,2025-09-20 08:43:21+00:00,METAR in effect: 202509200820 METAR COR LTFM 2...
